In [7]:
import heapq
from typing import List, Tuple, Optional, Set, Dict

State = Tuple[int, ...]

class PuzzleNode:
    __slots__ = ('state', 'empty_idx', 'g_cost', 'h_cost', 'f_cost', 'parent')

    def __init__(self, state: State, empty_idx: int, g_cost: int, h_cost: int, parent: Optional['PuzzleNode'] = None):
        self.state = state
        self.empty_idx = empty_idx
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = g_cost + h_cost
        self.parent = parent

    def __lt__(self, other: 'PuzzleNode') -> bool:
        if self.f_cost == other.f_cost:
            return self.h_cost < other.h_cost
        return self.f_cost < other.f_cost

class Puzzle15Solver:
    def __init__(self, initial_matrix: List[List[int]], goal_matrix: List[List[int]]):
        self.size = 4
        self.initial_state = self._flatten(initial_matrix)
        self.goal_state = self._flatten(goal_matrix)

        self.target_positions: Dict[int, Tuple[int, int]] = {
            val: (i // self.size, i % self.size)
            for i, val in enumerate(self.goal_state) if val != 0
        }

        self.moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def _flatten(self, matrix: List[List[int]]) -> State:
        return tuple(item for row in matrix for item in row)

    def _get_parity(self, state: State) -> int:
        inversions = 0
        state_no_zero = [x for x in state if x != 0]

        for i in range(len(state_no_zero)):
            for j in range(i + 1, len(state_no_zero)):
                if state_no_zero[i] > state_no_zero[j]:
                    inversions += 1

        empty_idx = state.index(0)
        empty_row_from_bottom = self.size - (empty_idx // self.size)

        return (inversions + empty_row_from_bottom) % 2

    def is_solvable(self) -> bool:
        return self._get_parity(self.initial_state) == self._get_parity(self.goal_state)

    def _manhattan_distance(self, state: State) -> int:
        distance = 0
        for i, val in enumerate(state):
            if val != 0:
                current_row, current_col = i // self.size, i % self.size
                target_row, target_col = self.target_positions[val]
                distance += abs(current_row - target_row) + abs(current_col - target_col)
        return distance

    def _get_neighbors(self, node: PuzzleNode) -> List[PuzzleNode]:
        neighbors = []
        r, c = node.empty_idx // self.size, node.empty_idx % self.size

        for dr, dc in self.moves:
            new_r, new_c = r + dr, c + dc

            if 0 <= new_r < self.size and 0 <= new_c < self.size:
                new_idx = new_r * self.size + new_c

                new_state_list = list(node.state)
                new_state_list[node.empty_idx], new_state_list[new_idx] = new_state_list[new_idx], new_state_list[node.empty_idx]
                new_state = tuple(new_state_list)

                g_cost = node.g_cost + 1
                h_cost = self._manhattan_distance(new_state)

                neighbors.append(PuzzleNode(new_state, new_idx, g_cost, h_cost, node))

        return neighbors

    def solve(self) -> Optional[PuzzleNode]:
        if not self.is_solvable():
            print("Cảnh báo: Không tồn tại lời giải cho bài toán này (Lỗi Parity).")
            return None

        initial_empty_idx = self.initial_state.index(0)
        initial_h = self._manhattan_distance(self.initial_state)
        root = PuzzleNode(self.initial_state, initial_empty_idx, 0, initial_h)

        open_list: List[PuzzleNode] = []
        heapq.heappush(open_list, root)

        closed_set: Set[State] = set()

        while open_list:
            current_node = heapq.heappop(open_list)

            if current_node.state == self.goal_state:
                return current_node

            closed_set.add(current_node.state)

            for neighbor in self._get_neighbors(current_node):
                if neighbor.state not in closed_set:
                    heapq.heappush(open_list, neighbor)

        return None

def print_solution(node: Optional[PuzzleNode]) -> None:
    if not node:
        return

    path = []
    while node:
        path.append(node)
        node = node.parent

    path.reverse()
    print(f"Hoàn thành trong {len(path) - 1} bước di chuyển.\n")

    for step, n in enumerate(path):
        print(f"Bước {step}:")
        for i in range(4):
            row = n.state[i*4 : (i+1)*4]
            print("\t".join(str(x) if x != 0 else "_" for x in row))
        print("-" * 20)

if __name__ == "__main__":
    initial_matrix = [
        [1, 2, 3, 4],
        [5, 6, 0, 8],
        [9, 10, 7, 11],
        [13, 14, 15, 12]
    ]

    final_matrix = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 0]
    ]

    solver = Puzzle15Solver(initial_matrix, final_matrix)
    solution = solver.solve()

    if solution:
        print_solution(solution)

Hoàn thành trong 3 bước di chuyển.

Bước 0:
1	2	3	4
5	6	_	8
9	10	7	11
13	14	15	12
--------------------
Bước 1:
1	2	3	4
5	6	7	8
9	10	_	11
13	14	15	12
--------------------
Bước 2:
1	2	3	4
5	6	7	8
9	10	11	_
13	14	15	12
--------------------
Bước 3:
1	2	3	4
5	6	7	8
9	10	11	12
13	14	15	_
--------------------


In [8]:
import heapq
from typing import Dict, List, Tuple, Callable, Optional, TypeVar, Generic

T = TypeVar('T')

class AStarGraph(Generic[T]):
    def __init__(self, adjacency_list: Dict[T, List[Tuple[T, float]]], heuristic_func: Optional[Callable[[T, T], float]] = None):
        self.graph = adjacency_list
        self.heuristic = heuristic_func if heuristic_func else self._zero_heuristic

    def _zero_heuristic(self, node_a: T, node_b: T) -> float:
        return 0.0

    def get_neighbors(self, node: T) -> List[Tuple[T, float]]:
        return self.graph.get(node, [])

    def find_path(self, start: T, goal: T) -> Optional[Tuple[List[T], float]]:
        if start not in self.graph or goal not in self.graph:
            return None

        open_set: List[Tuple[float, T]] = []
        heapq.heappush(open_set, (0.0, start))

        came_from: Dict[T, T] = {}

        g_score: Dict[T, float] = {node: float('inf') for node in self.graph}
        g_score[start] = 0.0

        while open_set:
            current_f, current_node = heapq.heappop(open_set)

            if current_node == goal:
                path = self._reconstruct_path(came_from, current_node)
                return path, g_score[goal]

            if current_f > g_score[current_node] + self.heuristic(current_node, goal):
                continue

            for neighbor, weight in self.get_neighbors(current_node):
                tentative_g_score = g_score[current_node] + weight

                if tentative_g_score < g_score.get(neighbor, float('inf')):
                    came_from[neighbor] = current_node
                    g_score[neighbor] = tentative_g_score
                    f_score = tentative_g_score + self.heuristic(neighbor, goal)
                    heapq.heappush(open_set, (f_score, neighbor))

        return None

    def _reconstruct_path(self, came_from: Dict[T, T], current: T) -> List[T]:
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        path.reverse()
        return path


if __name__ == "__main__":
    graph_data = {
        'A': [('B', 4), ('C', 2)],
        'B': [('A', 4), ('C', 1), ('D', 5)],
        'C': [('A', 2), ('B', 1), ('D', 8), ('E', 10)],
        'D': [('B', 5), ('C', 8), ('E', 2), ('Z', 6)],
        'E': [('C', 10), ('D', 2), ('Z', 3)],
        'Z': [('D', 6), ('E', 3)]
    }

    def custom_heuristic(node1: str, node2: str) -> float:
        h_values = {
            'A': 10, 'B': 8, 'C': 5,
            'D': 7, 'E': 3, 'Z': 0
        }
        return h_values.get(node1, 0)

    solver = AStarGraph(graph_data, heuristic_func=custom_heuristic)

    start_node = 'A'
    goal_node = 'Z'

    result = solver.find_path(start_node, goal_node)

    if result:
        path, total_cost = result
        print(f"Path: {' -> '.join(path)}")
        print(f"Total Cost: {total_cost}")
    else:
        print("No path found.")

Path: A -> C -> B -> D -> E -> Z
Total Cost: 13.0
